> **Not re-run for the final submission.** Parsing ~3.4 GB of zipped FAERS XML takes hours; its output, `data/eda_start/reports_level.parquet`, is the starting point for notebook 2. Outputs below are from the original run on Deepnote. Raw zips: FDA FAERS Quarterly Data Extract Files (XML), https://fis.fda.gov/extensions/FPD-QDE-FAERS/FPD-QDE-FAERS.html

# Deepnote starting dataset

This notebook turns the FAERS quarter zips in the project files into one table other notebooks can explore. It stops at that table. Cleaning beyond the codes you need to read the data, feature engineering, and EDA belong in the later notebooks.

Upload files named `faers_xml_*.zip` into the Deepnote `FAERS` folder (for example `faers_xml_2026q1.zip`). The name match is case-insensitive, so `2026Q2` works too. Each new quarter is picked up the next time this notebook rebuilds the table.

FAERS cases are nested. One report can list several drugs and several reactions. This notebook flattens that only far enough to produce **one row per case**, then keeps the latest `safetyreportversion` when the same case shows up in more than one quarter.

The Deepnote machine here has 5 GB of memory, and each quarter is a few gigabytes of XML. The parser streams each XML member inside the zip and writes small parquet chunks. It does not unzip the quarter onto disk and it does not load the full drug or reaction tables into memory.


## 1. Setup

`REPARSE` controls the expensive step. Leave it `False` to reload `data/eda_start/reports_level.parquet`. Set it to `True` after you add or replace a zip, then run the notebook from the top.

`APPLY_DELETED` drops case IDs that FDA listed in that quarter's `Deleted` file. `CHUNK_SIZE` is how many reports stay in memory before a parquet part is written. 10,000 is sized for this 5 GB machine.


In [1]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyarrow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])

import shutil
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
ZIP_DIR = PROJECT_ROOT / "FAERS"
ARCHIVE_PATH = PROJECT_ROOT / "FAERS.zip"


def quarter_zips_present(folder: Path) -> bool:
    if not folder.is_dir():
        return False
    return any(
        path.is_file() and path.suffix.lower() == ".zip" and path.name.lower().startswith("faers_xml_")
        for path in folder.iterdir()
    )


if quarter_zips_present(ZIP_DIR):
    print(f"using quarter zips in {ZIP_DIR}")
elif ARCHIVE_PATH.is_file():
    print(f"extracting {ARCHIVE_PATH.name} into {PROJECT_ROOT} ...", flush=True)
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(PROJECT_ROOT)
    print(f"extracted quarter zips into {ZIP_DIR}")
else:
    raise FileNotFoundError(
        "No faers_xml_*.zip files in the FAERS folder and no FAERS.zip in the working directory. "
        "Put FAERS.zip next to this notebook, or extract it so FAERS/ contains the quarter zips."
    )

APPLY_DELETED = True
REPARSE = False
CHUNK_SIZE = 10_000

OUT_DIR = PROJECT_ROOT / "data" / "eda_start"
PARTS_DIR = OUT_DIR / "parts"
REPORTS_DIR = PARTS_DIR / "reports"
DRUGS_DIR = PARTS_DIR / "drugs"
REACTIONS_DIR = PARTS_DIR / "reactions"
REPORTS_LEVEL_PATH = OUT_DIR / "reports_level.parquet"
MANIFEST_PATH = OUT_DIR / "source_zips.txt"
COLLAPSED_DIR = OUT_DIR / "collapsed"
COMPLETED_PATH = OUT_DIR / "completed_members.txt"
FAILED_PATH = OUT_DIR / "failed_members.txt"

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"ZIP_DIR = {ZIP_DIR}")
print(f"REPARSE = {REPARSE}  APPLY_DELETED = {APPLY_DELETED}  CHUNK_SIZE = {CHUNK_SIZE}")
print(f"output = {REPORTS_LEVEL_PATH}")
print(f"checkpoints = {COLLAPSED_DIR}")


using quarter zips in /datasets/_deepnote_work/FAERS
PROJECT_ROOT = /datasets/_deepnote_work
ZIP_DIR = /datasets/_deepnote_work/FAERS
REPARSE = False  APPLY_DELETED = True  CHUNK_SIZE = 10000
output = /datasets/_deepnote_work/data/eda_start/reports_level.parquet
checkpoints = /datasets/_deepnote_work/data/eda_start/collapsed


## 2. Which zips are uploaded?

The notebook only looks in the Deepnote `FAERS` folder. It does not extract the archives. Inside each zip it keeps ADR XML files (the `XML/` folder) and the deleted-ID list.

Add another quarter by uploading `faers_xml_*.zip` into `FAERS`. A zip with no ADR XML fails here, before any parsing starts.


In [2]:
def human_bytes(n: int) -> str:
    value = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if value < 1024 or unit == "GB":
            return f"{value:,.1f} {unit}" if unit != "B" else f"{int(value)} B"
        value /= 1024
    return f"{n} B"


def find_quarter_zips(root: Path) -> list[Path]:
    if not root.is_dir():
        return []
    zips = [
        path
        for path in root.iterdir()
        if path.is_file()
        and path.suffix.lower() == ".zip"
        and path.name.lower().startswith("faers_xml_")
    ]
    return sorted(zips, key=lambda path: path.name.lower())


def classify_member(name: str) -> str | None:
    parts = [part.lower() for part in name.replace("\\", "/").split("/") if part]
    if not parts:
        return None
    filename = parts[-1]
    parents = parts[:-1]
    if filename.endswith(".xml") and "xml" in parents:
        return "xml"
    if filename.startswith("delete") and filename.endswith(".txt"):
        return "deleted"
    return None


def list_zip_members(zip_path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue
            kind = classify_member(info.filename)
            if kind is None:
                continue
            rows.append(
                {
                    "zip": zip_path.name,
                    "quarter": zip_path.stem,
                    "member": info.filename,
                    "kind": kind,
                    "size": info.file_size,
                }
            )
    return rows


zip_paths = find_quarter_zips(ZIP_DIR)
if not zip_paths:
    raise FileNotFoundError(
        "No faers_xml_*.zip files in the FAERS folder. "
        "Put FAERS.zip in the working directory, or add faers_xml_*.zip files to the FAERS folder, and run from the top."
    )

member_rows: list[dict[str, Any]] = []
for zip_path in zip_paths:
    member_rows.extend(list_zip_members(zip_path))

inventory = pd.DataFrame(member_rows)
if inventory.empty or not (inventory["kind"] == "xml").any():
    raise FileNotFoundError(
        "Found quarter zips, but none contain an XML/*.xml ADR file. "
        "Check that the uploads are the FDA XML quarterly extracts."
    )

inventory["size_human"] = inventory["size"].map(human_bytes)
display(inventory[["quarter", "kind", "member", "size_human"]].sort_values(["quarter", "kind", "member"]))

xml_members = inventory[inventory["kind"] == "xml"]
deleted_members = inventory[inventory["kind"] == "deleted"]
print(f"{len(zip_paths)} zip(s), {len(xml_members)} XML file(s), {human_bytes(int(xml_members['size'].sum()))} uncompressed")
print(f"{len(deleted_members)} deleted-ID file(s)")

current_manifest = "\n".join(f"{path.name}\t{path.stat().st_size}" for path in zip_paths)
if REPORTS_LEVEL_PATH.exists() and MANIFEST_PATH.exists() and not REPARSE:
    saved_manifest = MANIFEST_PATH.read_text(encoding="utf-8").strip()
    if saved_manifest != current_manifest:
        print(
            "The uploaded zips differ from the ones used to build reports_level.parquet. "
            "Set REPARSE = True and run from the top to include them."
        )


,quarter,kind,member,size_human
0,faers_xml_2020Q1,xml,XML/1_ADR20Q1.xml,689.4 MB
1,faers_xml_2020Q1,xml,XML/2_ADR20Q1.xml,656.2 MB
2,faers_xml_2020Q1,xml,XML/3_ADR20Q1.xml,796.7 MB
3,faers_xml_2020Q2,xml,XML/1_ADR20Q2.xml,635.4 MB
4,faers_xml_2020Q2,xml,XML/2_ADR20Q2.xml,637.0 MB
...,...,...,...,...
89,faers_xml_2026q1,xml,XML/3_ADR26Q1.xml,747.9 MB
93,faers_xml_2026q2,deleted,Deleted/DELETE26Q2.txt,37.1 KB
94,faers_xml_2026q2,xml,XML/1_ADR26Q2.xml,698.1 MB
96,faers_xml_2026q2,xml,XML/2_ADR26Q2.xml,651.0 MB


26 zip(s), 78 XML file(s), 56.3 GB uncompressed
19 deleted-ID file(s)


## 3. Stream one report at a time

`ET.parse` would load an entire XML file. These files are hundreds of megabytes each, so `iterparse` yields one `</safetyreport>` and then drops it. Reading that element from `ZipFile.open` means the XML is never written out as its own file.

Only a short field list is kept: who reported the case, how serious it was, age and sex, each drug's role and ingredient, and each reaction's preferred term.


In [3]:
REPORT_TAGS = [
    "safetyreportversion",
    "safetyreportid",
    "primarysourcecountry",
    "occurcountry",
    "reporttype",
    "serious",
    "seriousnessdeath",
    "seriousnesshospitalization",
    "receivedate",
    "receiptdate",
    "fulfillexpeditecriteria",
]

PATIENT_TAGS = [
    "patientonsetage",
    "patientonsetageunit",
    "patientagegroup",
    "patientweight",
    "patientsex",
]

DRUG_TAGS = [
    "drugcharacterization",
    "medicinalproduct",
    "drugindication",
    "drugadministrationroute",
]


def local_tag(tag: str) -> str:
    return tag.split("}")[-1]


def child_text(parent: ET.Element | None, tag: str) -> str | None:
    if parent is None:
        return None
    node = parent.find(tag)
    if node is None or node.text is None:
        return None
    text = node.text.strip()
    return text if text else None


def flatten_report(elem: ET.Element, quarter: str, report_uid: int) -> tuple[dict[str, Any], list[dict[str, Any]], list[dict[str, Any]]]:
    report: dict[str, Any] = {"report_uid": report_uid, "source_quarter": quarter}
    for tag in REPORT_TAGS:
        report[tag] = child_text(elem, tag)

    primary = elem.find("primarysource")
    report["qualification"] = child_text(primary, "qualification")

    patient = elem.find("patient")
    for tag in PATIENT_TAGS:
        report[tag] = child_text(patient, tag)

    drugs: list[dict[str, Any]] = []
    reactions: list[dict[str, Any]] = []
    if patient is not None:
        for i, drug in enumerate(patient.findall("drug"), start=1):
            row: dict[str, Any] = {"report_uid": report_uid, "drug_seq": i}
            for tag in DRUG_TAGS:
                row[tag] = child_text(drug, tag)
            substance = drug.find("activesubstance")
            row["activesubstancename"] = child_text(substance, "activesubstancename")
            drugs.append(row)
        for i, reaction in enumerate(patient.findall("reaction"), start=1):
            reactions.append(
                {
                    "report_uid": report_uid,
                    "reaction_seq": i,
                    "reactionmeddrapt": child_text(reaction, "reactionmeddrapt"),
                    "reactionoutcome": child_text(reaction, "reactionoutcome"),
                }
            )
    return report, drugs, reactions


def iter_safety_reports(stream):
    root = None
    for event, elem in ET.iterparse(stream, events=("start", "end")):
        if event == "start":
            if root is None:
                root = elem
            continue
        if local_tag(elem.tag) != "safetyreport":
            continue
        yield elem
        elem.clear()
        if root is not None:
            root.clear()


def load_deleted_ids(paths: list[Path]) -> set[str]:
    deleted: set[str] = set()
    for zip_path in paths:
        with zipfile.ZipFile(zip_path) as archive:
            for info in archive.infolist():
                if info.is_dir() or classify_member(info.filename) != "deleted":
                    continue
                text = archive.read(info).decode("utf-8", errors="replace")
                for line in text.splitlines():
                    value = line.strip()
                    if value:
                        deleted.add(value)
    return deleted


print(
    f"Keeping {len(REPORT_TAGS)} report tags, {len(PATIENT_TAGS)} patient tags, "
    f"{len(DRUG_TAGS)} drug tags, plus ingredient and reaction PT."
)


Keeping 11 report tags, 5 patient tags, 4 drug tags, plus ingredient and reaction PT.


## 4. Parse every uploaded quarter

If `reports_level.parquet` is already there and `REPARSE` is False, this cell does not read the XML. When it does parse, it writes part files under `data/eda_start/parts/` and clears the Python lists after each chunk.

A case ID that appears in any quarter's delete file is skipped. The same case can still appear in two quarters; the next section keeps the newer version.


In [4]:
import pyarrow as pa
import pyarrow.parquet as pq

SEX = {"0": "unknown", "1": "male", "2": "female"}
QUALIFICATION = {
    "1": "physician",
    "2": "pharmacist",
    "3": "other_health_professional",
    "4": "lawyer",
    "5": "consumer_or_non_health_professional",
}
AGE_TO_YEARS = {
    "800": 10.0,
    "801": 1.0,
    "802": 1.0 / 12.0,
    "803": 1.0 / 52.1775,
    "804": 1.0 / 365.25,
    "805": 1.0 / (365.25 * 24.0),
}

LEVEL_COLUMNS = [
    "source_quarter",
    "safetyreportversion",
    "safetyreportid",
    "primarysourcecountry",
    "occurcountry",
    "reporttype",
    "serious",
    "seriousnessdeath",
    "seriousnesshospitalization",
    "receivedate",
    "receiptdate",
    "fulfillexpeditecriteria",
    "qualification",
    "patientonsetage",
    "patientonsetageunit",
    "patientagegroup",
    "patientweight",
    "patientsex",
    "age_years",
    "patientweight_kg",
    "receivedate_parsed",
    "death",
    "hospitalization",
    "y_hosp_or_death",
    "patientsex_label",
    "qualification_label",
    "n_drugs",
    "n_suspect_drugs",
    "suspect_ingredients",
    "n_reactions",
    "reaction_pts",
]


def yesno(series: pd.Series) -> pd.Series:
    mapped = series.astype("string").str.strip()
    out = pd.Series(pd.NA, index=series.index, dtype="boolean")
    out = out.mask(mapped.eq("1"), True)
    out = out.mask(mapped.eq("2"), False)
    return out


def age_years(age: Any, unit: Any) -> float:
    if pd.isna(age) or pd.isna(unit):
        return np.nan
    try:
        value = float(str(age).strip())
    except ValueError:
        return np.nan
    factor = AGE_TO_YEARS.get(str(unit).strip())
    if factor is None:
        return np.nan
    years = value * factor
    if years < 0 or years > 120:
        return np.nan
    return years


def parse_day(value: Any) -> pd.Timestamp:
    if pd.isna(value):
        return pd.NaT
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    if len(digits) < 8:
        return pd.NaT
    try:
        return pd.Timestamp(year=int(digits[:4]), month=int(digits[4:6]), day=int(digits[6:8]))
    except ValueError:
        return pd.NaT


def clean_name(value: Any) -> str | None:
    if pd.isna(value):
        return None
    text = " ".join(str(value).upper().split())
    return text or None


def pipe_join(values: pd.Series):
    cleaned = values.dropna().astype("string").str.strip()
    cleaned = cleaned[cleaned.ne("")].drop_duplicates()
    if cleaned.empty:
        return pd.NA
    return " | ".join(cleaned.tolist())


def clean_reports(raw: pd.DataFrame) -> pd.DataFrame:
    reports = raw.copy()
    reports["safetyreportversion"] = pd.to_numeric(reports["safetyreportversion"], errors="coerce").astype("Int64")
    reports["age_years"] = [
        age_years(age, unit)
        for age, unit in zip(reports["patientonsetage"], reports["patientonsetageunit"])
    ]
    reports["patientweight_kg"] = pd.to_numeric(reports["patientweight"], errors="coerce")
    reports["receivedate_parsed"] = reports["receivedate"].map(parse_day)
    reports["death"] = yesno(reports["seriousnessdeath"])
    reports["hospitalization"] = yesno(reports["seriousnesshospitalization"])
    reports["y_hosp_or_death"] = reports["death"].fillna(False) | reports["hospitalization"].fillna(False)
    both_missing = reports["death"].isna() & reports["hospitalization"].isna()
    reports.loc[both_missing, "y_hosp_or_death"] = pd.NA
    reports["y_hosp_or_death"] = reports["y_hosp_or_death"].astype("boolean")
    reports["patientsex_label"] = reports["patientsex"].astype("string").str.strip().map(SEX)
    reports["qualification_label"] = reports["qualification"].astype("string").str.strip().map(QUALIFICATION)
    return reports


def collapse_chunk(reports_raw: pd.DataFrame, drugs_raw: pd.DataFrame, reactions_raw: pd.DataFrame) -> pd.DataFrame:
    reports = clean_reports(reports_raw)

    if drugs_raw.empty:
        drug_counts = pd.DataFrame(columns=["report_uid", "n_drugs", "n_suspect_drugs"])
        suspect_lists = pd.DataFrame(columns=["report_uid", "suspect_ingredients"])
    else:
        drugs = drugs_raw.copy()
        drugs["is_suspect"] = drugs["drugcharacterization"].astype("string").str.strip().eq("1")
        drugs["ingredient"] = [
            clean_name(substance) or clean_name(product)
            for substance, product in zip(drugs["activesubstancename"], drugs["medicinalproduct"])
        ]
        drug_counts = drugs.groupby("report_uid", as_index=False).agg(
            n_drugs=("drug_seq", "size"),
            n_suspect_drugs=("is_suspect", "sum"),
        )
        suspect_lists = (
            drugs.loc[drugs["is_suspect"]]
            .groupby("report_uid", as_index=False)
            .agg(suspect_ingredients=("ingredient", pipe_join))
        )

    if reactions_raw.empty:
        reaction_lists = pd.DataFrame(columns=["report_uid", "n_reactions", "reaction_pts"])
    else:
        reactions = reactions_raw.copy()
        reactions["reaction_pt"] = reactions["reactionmeddrapt"].map(clean_name)
        reaction_lists = reactions.groupby("report_uid", as_index=False).agg(
            n_reactions=("reaction_seq", "size"),
            reaction_pts=("reaction_pt", pipe_join),
        )

    merged = (
        reports.merge(drug_counts, on="report_uid", how="left")
        .merge(suspect_lists, on="report_uid", how="left")
        .merge(reaction_lists, on="report_uid", how="left")
    )
    for col in ("n_drugs", "n_suspect_drugs", "n_reactions"):
        merged[col] = merged[col].fillna(0).astype("Int64")
    return merged



def xml_jobs() -> list[tuple[Path, str, str]]:
    jobs: list[tuple[Path, str, str]] = []
    xml_inventory = inventory[inventory["kind"] == "xml"].sort_values(["quarter", "member"])
    for row in xml_inventory.itertuples(index=False):
        jobs.append((ZIP_DIR / row.zip, row.member, row.quarter))
    return jobs


def member_key(zip_name: str, member: str) -> str:
    return f"{zip_name}\t{member}"


def member_slug(zip_name: str, member: str) -> str:
    safe_member = member.replace("\\", "/").strip("/").replace("/", "__")
    return f"{zip_name}__{safe_member}"


def load_member_keys(path: Path) -> set[str]:
    if not path.exists():
        return set()
    return {line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()}


def append_key(path: Path, key: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(key + "\n")
        handle.flush()


def compression_supported(zip_path: Path, member: str) -> bool:
    with zipfile.ZipFile(zip_path) as archive:
        info = archive.getinfo(member)
    return info.compress_type in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)


def clear_checkpoints() -> None:
    for folder in (PARTS_DIR, COLLAPSED_DIR):
        if folder.exists():
            shutil.rmtree(folder)
    for path in (COMPLETED_PATH, FAILED_PATH, REPORTS_LEVEL_PATH):
        if path.exists():
            path.unlink()


def write_chunk(
    reports: list[dict[str, Any]],
    drugs: list[dict[str, Any]],
    reactions: list[dict[str, Any]],
    part: int,
    report_dir: Path,
    drug_dir: Path,
    reaction_dir: Path,
) -> None:
    for folder in (report_dir, drug_dir, reaction_dir):
        folder.mkdir(parents=True, exist_ok=True)
    suffix = f"part_{part:04d}.parquet"
    frames = {
        report_dir / suffix: pd.DataFrame(reports),
        drug_dir / suffix: pd.DataFrame(drugs),
        reaction_dir / suffix: pd.DataFrame(reactions),
    }
    for path, frame in frames.items():
        frame.to_parquet(path, index=False)


def collapse_part_dirs(report_dir: Path, drug_dir: Path, reaction_dir: Path, dest: Path) -> int:
    report_paths = sorted(report_dir.glob("part_*.parquet"))
    if not report_paths:
        return 0
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        dest.unlink()
    writer = None
    total = 0
    try:
        for report_path in report_paths:
            drugs_path = drug_dir / report_path.name
            reactions_path = reaction_dir / report_path.name
            piece = collapse_chunk(
                pd.read_parquet(report_path),
                pd.read_parquet(drugs_path) if drugs_path.exists() else pd.DataFrame(),
                pd.read_parquet(reactions_path) if reactions_path.exists() else pd.DataFrame(),
            )
            piece = piece.reindex(columns=LEVEL_COLUMNS)
            if piece.empty:
                continue
            table = pa.Table.from_pandas(piece, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(dest, table.schema)
            else:
                table = table.cast(writer.schema, safe=False)
            writer.write_table(table)
            total += len(piece)
    finally:
        if writer is not None:
            writer.close()
    if total == 0 and dest.exists():
        dest.unlink()
    return total


def legacy_report_parts() -> list[Path]:
    if not REPORTS_DIR.exists():
        return []
    return sorted(REPORTS_DIR.glob("part_*.parquet"))


def recover_raw_parts(jobs: list[tuple[Path, str, str]]) -> None:
    if REPARSE or COMPLETED_PATH.exists():
        return
    parts = legacy_report_parts()
    if not parts:
        return
    print(f"collapsing {len(parts)} raw part(s) from the interrupted run ...", flush=True)
    n_rows = collapse_part_dirs(REPORTS_DIR, DRUGS_DIR, REACTIONS_DIR, COLLAPSED_DIR / "recovered.parquet")
    print(f"wrote recovered.parquet  rows={n_rows:,}", flush=True)
    cutoff = None
    for zip_path, member, _quarter in jobs:
        if not compression_supported(zip_path, member):
            cutoff = member_key(zip_path.name, member)
            break
    if cutoff is None:
        print(
            "No XML member uses an unsupported compression type, so earlier members were not marked complete. "
            "The recovered rows stay in recovered.parquet."
        )
    else:
        for zip_path, member, _quarter in jobs:
            key = member_key(zip_path.name, member)
            if key == cutoff:
                break
            append_key(COMPLETED_PATH, key)
        print(f"marked members before {cutoff} complete", flush=True)
    for folder in (REPORTS_DIR, DRUGS_DIR, REACTIONS_DIR):
        if not folder.exists():
            continue
        for path in folder.glob("part_*.parquet"):
            path.unlink()


def parse_member(zip_path: Path, member: str, quarter: str, deleted: set[str]) -> tuple[int, int, int]:
    slug = member_slug(zip_path.name, member)
    member_root = PARTS_DIR / slug
    report_dir = member_root / "reports"
    drug_dir = member_root / "drugs"
    reaction_dir = member_root / "reactions"
    if member_root.exists():
        shutil.rmtree(member_root)
    reports: list[dict[str, Any]] = []
    drugs: list[dict[str, Any]] = []
    reactions: list[dict[str, Any]] = []
    n_seen = 0
    n_kept = 0
    n_deleted = 0
    part = 0
    report_uid = 0
    try:
        if not compression_supported(zip_path, member):
            raise NotImplementedError(
                f"That compression method is not supported ({zip_path.name} :: {member})"
            )
        with zipfile.ZipFile(zip_path) as archive:
            with archive.open(member) as stream:
                for elem in iter_safety_reports(stream):
                    n_seen += 1
                    report, drug_rows, reaction_rows = flatten_report(elem, quarter, report_uid)
                    report_uid += 1
                    if APPLY_DELETED and report["safetyreportid"] in deleted:
                        n_deleted += 1
                        continue
                    reports.append(report)
                    drugs.extend(drug_rows)
                    reactions.extend(reaction_rows)
                    n_kept += 1
                    if len(reports) >= CHUNK_SIZE:
                        write_chunk(reports, drugs, reactions, part, report_dir, drug_dir, reaction_dir)
                        print(
                            f"  wrote part {part:04d}  kept={n_kept:,}  skipped_deleted={n_deleted:,}",
                            flush=True,
                        )
                        part += 1
                        reports, drugs, reactions = [], [], []
        if reports:
            write_chunk(reports, drugs, reactions, part, report_dir, drug_dir, reaction_dir)
            print(
                f"  wrote part {part:04d}  kept={n_kept:,}  skipped_deleted={n_deleted:,}",
                flush=True,
            )
        n_rows = collapse_part_dirs(report_dir, drug_dir, reaction_dir, COLLAPSED_DIR / f"{slug}.parquet")
    except NotImplementedError:
        if member_root.exists():
            shutil.rmtree(member_root)
        raise
    if member_root.exists():
        shutil.rmtree(member_root)
    append_key(COMPLETED_PATH, member_key(zip_path.name, member))
    print(f"checkpoint {zip_path.name} :: {member}  rows={n_rows:,}", flush=True)
    return n_seen, n_kept, n_deleted


def extract_all() -> dict[str, int]:
    if REPARSE:
        clear_checkpoints()
    jobs = xml_jobs()
    recover_raw_parts(jobs)
    if FAILED_PATH.exists():
        FAILED_PATH.unlink()
    deleted = load_deleted_ids(zip_paths) if APPLY_DELETED else set()
    completed = load_member_keys(COMPLETED_PATH)
    n_seen = 0
    n_kept = 0
    n_deleted = 0
    n_skipped = 0
    n_failed = 0
    for zip_path, member, quarter in jobs:
        key = member_key(zip_path.name, member)
        if key in completed:
            n_skipped += 1
            print(f"skipping {zip_path.name} :: {member}", flush=True)
            continue
        print(f"reading {zip_path.name} :: {member} ...", flush=True)
        try:
            seen, kept, dropped = parse_member(zip_path, member, quarter, deleted)
        except NotImplementedError as exc:
            n_failed += 1
            append_key(FAILED_PATH, key)
            print(f"failed {zip_path.name} :: {member}: {exc}", flush=True)
            continue
        n_seen += seen
        n_kept += kept
        n_deleted += dropped
    return {
        "xml_files": len(jobs),
        "members_skipped": n_skipped,
        "members_failed": n_failed,
        "reports_seen": n_seen,
        "reports_kept": n_kept,
        "deleted_ids_loaded": len(deleted),
        "deleted_skipped": n_deleted,
    }


jobs = xml_jobs()
completed_now = set() if REPARSE else load_member_keys(COMPLETED_PATH)
pending = [job for job in jobs if member_key(job[0].name, job[1]) not in completed_now]
need_parse = REPARSE or bool(pending) or bool(legacy_report_parts()) or not REPORTS_LEVEL_PATH.exists()
if need_parse:
    parse_stats = extract_all()
    display(pd.Series(parse_stats, name="value").to_frame())
else:
    print(
        f"Skipping XML. {len(completed_now)} member(s) already checkpointed "
        f"and {REPORTS_LEVEL_PATH.name} exists. Set REPARSE = True to rebuild from scratch."
    )


skipping faers_xml_2020Q1.zip :: XML/1_ADR20Q1.xml
skipping faers_xml_2020Q1.zip :: XML/2_ADR20Q1.xml
skipping faers_xml_2020Q1.zip :: XML/3_ADR20Q1.xml
skipping faers_xml_2020Q2.zip :: XML/1_ADR20Q2.xml
skipping faers_xml_2020Q2.zip :: XML/2_ADR20Q2.xml
skipping faers_xml_2020Q2.zip :: XML/3_ADR20Q2.xml
skipping faers_xml_2020Q3.zip :: XML/1_ADR20Q3.xml
skipping faers_xml_2020Q3.zip :: XML/2_ADR20Q3.xml
skipping faers_xml_2020Q3.zip :: XML/3_ADR20Q3.xml
skipping faers_xml_2020Q4.zip :: XML/1_ADR20Q4.xml
skipping faers_xml_2020Q4.zip :: XML/2_ADR20Q4.xml
skipping faers_xml_2020Q4.zip :: XML/3_ADR20Q4.xml
skipping faers_xml_2021Q1.zip :: XML/1_ADR21Q1_format.xml
skipping faers_xml_2021Q1.zip :: XML/2_ADR21Q1_format.xml
skipping faers_xml_2021Q1.zip :: XML/3_ADR21Q1_format.xml
skipping faers_xml_2021Q2.zip :: XML/1_ADR21Q2.xml
skipping faers_xml_2021Q2.zip :: XML/2_ADR21Q2.xml
skipping faers_xml_2021Q2.zip :: XML/3_ADR21Q2.xml
skipping faers_xml_2021Q3.zip :: XML/1_ADR21Q3.xml
skipping f

,value
xml_files,78
members_skipped,69
members_failed,9
reports_seen,0
reports_kept,0
deleted_ids_loaded,95602
deleted_skipped,0


## 5. Light clean, then one row per case

Each chunk is cleaned and collapsed on its own, then thrown away. The long drug and reaction tables are never concatenated.

The codes below are the ones that are easy to misread:

- Seriousness fields use **1 = yes** and **2 = no**.
- Age units are converted to years. Values outside 0–120 years are treated as missing.
- Dates become a day only when the source has a full `YYYYMMDD`. Year-only and year-month values stay missing.
- The ingredient string is the active substance when FDA sent one, otherwise the product name. Both are stripped and upper-cased.
- Suspect drugs are `drugcharacterization` 1.

After the chunks are combined, the latest `safetyreportversion` is kept for each `safetyreportid`. A case that was updated in a later quarter does not count twice.


In [5]:
SEX = {"0": "unknown", "1": "male", "2": "female"}
QUALIFICATION = {
    "1": "physician",
    "2": "pharmacist",
    "3": "other_health_professional",
    "4": "lawyer",
    "5": "consumer_or_non_health_professional",
}
AGE_TO_YEARS = {
    "800": 10.0,
    "801": 1.0,
    "802": 1.0 / 12.0,
    "803": 1.0 / 52.1775,
    "804": 1.0 / 365.25,
    "805": 1.0 / (365.25 * 24.0),
}

LEVEL_COLUMNS = [
    "source_quarter",
    "safetyreportversion",
    "safetyreportid",
    "primarysourcecountry",
    "occurcountry",
    "reporttype",
    "serious",
    "seriousnessdeath",
    "seriousnesshospitalization",
    "receivedate",
    "receiptdate",
    "fulfillexpeditecriteria",
    "qualification",
    "patientonsetage",
    "patientonsetageunit",
    "patientagegroup",
    "patientweight",
    "patientsex",
    "age_years",
    "patientweight_kg",
    "receivedate_parsed",
    "death",
    "hospitalization",
    "y_hosp_or_death",
    "patientsex_label",
    "qualification_label",
    "n_drugs",
    "n_suspect_drugs",
    "suspect_ingredients",
    "n_reactions",
    "reaction_pts",
]


def yesno(series: pd.Series) -> pd.Series:
    mapped = series.astype("string").str.strip()
    out = pd.Series(pd.NA, index=series.index, dtype="boolean")
    out = out.mask(mapped.eq("1"), True)
    out = out.mask(mapped.eq("2"), False)
    return out


def age_years(age: Any, unit: Any) -> float:
    if pd.isna(age) or pd.isna(unit):
        return np.nan
    try:
        value = float(str(age).strip())
    except ValueError:
        return np.nan
    factor = AGE_TO_YEARS.get(str(unit).strip())
    if factor is None:
        return np.nan
    years = value * factor
    if years < 0 or years > 120:
        return np.nan
    return years


def parse_day(value: Any) -> pd.Timestamp:
    if pd.isna(value):
        return pd.NaT
    digits = "".join(ch for ch in str(value) if ch.isdigit())
    if len(digits) < 8:
        return pd.NaT
    try:
        return pd.Timestamp(year=int(digits[:4]), month=int(digits[4:6]), day=int(digits[6:8]))
    except ValueError:
        return pd.NaT


def clean_name(value: Any) -> str | None:
    if pd.isna(value):
        return None
    text = " ".join(str(value).upper().split())
    return text or None


def pipe_join(values: pd.Series):
    cleaned = values.dropna().astype("string").str.strip()
    cleaned = cleaned[cleaned.ne("")].drop_duplicates()
    if cleaned.empty:
        return pd.NA
    return " | ".join(cleaned.tolist())


def clean_reports(raw: pd.DataFrame) -> pd.DataFrame:
    reports = raw.copy()
    reports["safetyreportversion"] = pd.to_numeric(reports["safetyreportversion"], errors="coerce").astype("Int64")
    reports["age_years"] = [
        age_years(age, unit)
        for age, unit in zip(reports["patientonsetage"], reports["patientonsetageunit"])
    ]
    reports["patientweight_kg"] = pd.to_numeric(reports["patientweight"], errors="coerce")
    reports["receivedate_parsed"] = reports["receivedate"].map(parse_day)
    reports["death"] = yesno(reports["seriousnessdeath"])
    reports["hospitalization"] = yesno(reports["seriousnesshospitalization"])
    reports["y_hosp_or_death"] = reports["death"].fillna(False) | reports["hospitalization"].fillna(False)
    both_missing = reports["death"].isna() & reports["hospitalization"].isna()
    reports.loc[both_missing, "y_hosp_or_death"] = pd.NA
    reports["y_hosp_or_death"] = reports["y_hosp_or_death"].astype("boolean")
    reports["patientsex_label"] = reports["patientsex"].astype("string").str.strip().map(SEX)
    reports["qualification_label"] = reports["qualification"].astype("string").str.strip().map(QUALIFICATION)
    return reports


def collapse_chunk(reports_raw: pd.DataFrame, drugs_raw: pd.DataFrame, reactions_raw: pd.DataFrame) -> pd.DataFrame:
    reports = clean_reports(reports_raw)

    if drugs_raw.empty:
        drug_counts = pd.DataFrame(columns=["report_uid", "n_drugs", "n_suspect_drugs"])
        suspect_lists = pd.DataFrame(columns=["report_uid", "suspect_ingredients"])
    else:
        drugs = drugs_raw.copy()
        drugs["is_suspect"] = drugs["drugcharacterization"].astype("string").str.strip().eq("1")
        drugs["ingredient"] = [
            clean_name(substance) or clean_name(product)
            for substance, product in zip(drugs["activesubstancename"], drugs["medicinalproduct"])
        ]
        drug_counts = drugs.groupby("report_uid", as_index=False).agg(
            n_drugs=("drug_seq", "size"),
            n_suspect_drugs=("is_suspect", "sum"),
        )
        suspect_lists = (
            drugs.loc[drugs["is_suspect"]]
            .groupby("report_uid", as_index=False)
            .agg(suspect_ingredients=("ingredient", pipe_join))
        )

    if reactions_raw.empty:
        reaction_lists = pd.DataFrame(columns=["report_uid", "n_reactions", "reaction_pts"])
    else:
        reactions = reactions_raw.copy()
        reactions["reaction_pt"] = reactions["reactionmeddrapt"].map(clean_name)
        reaction_lists = reactions.groupby("report_uid", as_index=False).agg(
            n_reactions=("reaction_seq", "size"),
            reaction_pts=("reaction_pt", pipe_join),
        )

    merged = (
        reports.merge(drug_counts, on="report_uid", how="left")
        .merge(suspect_lists, on="report_uid", how="left")
        .merge(reaction_lists, on="report_uid", how="left")
    )
    for col in ("n_drugs", "n_suspect_drugs", "n_reactions"):
        merged[col] = merged[col].fillna(0).astype("Int64")
    return merged


def build_reports_level() -> pd.DataFrame:
    collapsed_paths = sorted(COLLAPSED_DIR.glob("*.parquet"))
    if not collapsed_paths:
        raise FileNotFoundError(
            f"No collapsed files in {COLLAPSED_DIR}. Run the parse cell first."
        )
    pieces: list[pd.DataFrame] = []
    for path in collapsed_paths:
        piece = pd.read_parquet(path)
        pieces.append(piece)
        print(f"loaded {path.name}  rows={len(piece):,}", flush=True)
    combined = pd.concat(pieces, ignore_index=True)
    before = len(combined)
    deduped = (
        combined.sort_values(
            ["safetyreportid", "safetyreportversion", "source_quarter"],
            na_position="first",
        )
        .groupby("safetyreportid", as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )
    print(f"rows before latest-version filter: {before:,}")
    print(f"rows after latest-version filter:  {len(deduped):,}")
    print(f"older versions dropped:            {before - len(deduped):,}")
    return deduped.reindex(columns=LEVEL_COLUMNS)


if COLLAPSED_DIR.exists() and any(COLLAPSED_DIR.glob("*.parquet")) and (need_parse or not REPORTS_LEVEL_PATH.exists()):
    reports_level = build_reports_level()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    reports_level.to_parquet(REPORTS_LEVEL_PATH, index=False)
    MANIFEST_PATH.write_text(current_manifest + "\n", encoding="utf-8")
    print(f"saved {REPORTS_LEVEL_PATH}")
    failed = load_member_keys(FAILED_PATH)
    if failed:
        print(f"failed members still excluded: {len(failed)}")
        for line in sorted(failed):
            print(f"  {line}")
elif REPORTS_LEVEL_PATH.exists():
    reports_level = pd.read_parquet(REPORTS_LEVEL_PATH)
    print(f"loaded {REPORTS_LEVEL_PATH}")
else:
    raise FileNotFoundError(
        "data/eda_start/reports_level.parquet is missing and there are no collapsed checkpoints."
    )


## 6. What the other notebooks should load

`reports_level` is one row per case. `suspect_ingredients` and `reaction_pts` are pipe-separated lists, so a plot of those columns is a summary of the case, not a drug-level or reaction-level table.

This extract is spontaneous reporting. Counts are not incidence: there is no exposure denominator, age is often missing, and a drug can look common because it is used often or reported often.

Other notebooks in this project can start from the saved file:

```python
import pandas as pd
reports = pd.read_parquet("data/eda_start/reports_level.parquet")
```


In [3]:
print("reports_level shape:", reports_level.shape)
print("quarters:")
display(reports_level["source_quarter"].value_counts(dropna=False).rename("n_cases").to_frame())
display(
    reports_level[
        [
            "safetyreportid",
            "safetyreportversion",
            "source_quarter",
            "receivedate_parsed",
            "age_years",
            "patientsex_label",
            "qualification_label",
            "n_suspect_drugs",
            "n_reactions",
            "suspect_ingredients",
            "reaction_pts",
            "death",
            "hospitalization",
            "y_hosp_or_death",
        ]
    ].head(8)
)
